# Lab 2 — Swap the black box: vector and hybrid search

**Companion to course Lesson 4.**

In Lab 1 you measured a lexical (BM25) retriever. Now we keep
everything else fixed — same dataset, same queries, same
metrics — and swap out the retrieval strategy. This is the
experimental discipline that lets you say something meaningful
when comparing systems: change one variable, hold everything
else constant.

## The three strategies

| name | aggregation stage | strength |
|---|---|---|
| **lexical (BM25)** | `$search` | exact terms, names, IDs |
| **vector** | `$vectorSearch` | semantic match, paraphrase |
| **hybrid** | `$rankFusion` | best of both, weighted fusion |

You'll see all three as MQL pipelines — no library indirection.

In [ ]:
import os, sys
# Notebooks live at the repo root; library modules live in src/.
_REPO_ROOT = os.path.abspath(os.getcwd())
_SRC = os.path.join(_REPO_ROOT, "src")
if _SRC not in sys.path:
    sys.path.insert(0, _SRC)

In [ ]:
from dotenv import load_dotenv
load_dotenv(os.path.join(_REPO_ROOT, ".env"))

import math, pymongo, voyageai
from lib import load_beir_dataset, embed_queries

DATASET           = "scifact"
DB_NAME           = "voyage_context_demo"
COLL_NAME         = f"chunks_{DATASET.replace('-', '_')}"
VECTOR_INDEX_NAME = "voyage_vector_index"
TEXT_INDEX_NAME   = "voyage_text_index"
MONGODB_BASE_URL  = "https://ai.mongodb.com/v1"
TOP_K             = 10
N_QUERIES         = 30

client = pymongo.MongoClient(os.environ["MONGODB_URI"])
coll   = client[DB_NAME][COLL_NAME]
corpus, queries, qrels, info = load_beir_dataset(DATASET)

voyage = voyageai.Client(
    api_key=os.environ["VOYAGE_API_KEY"],
    base_url=MONGODB_BASE_URL,
)
print(f"Connected to {DB_NAME}.{COLL_NAME}, {len(queries)} test queries available.")

## Step 1 — Embed the queries with `voyage-3-large`

For retrieval, we embed the user's query with a query-side
model and compare it to the chunk vectors we stored in Lab 0.
`voyage-context-3` is a *document* embedder — for queries we
use the matching standard model, `voyage-3-large`.

The embedding helper just wraps `voyage.embed(...)` — we use
it directly here.

In [ ]:
query_ids = list(queries.keys())[:N_QUERIES]
q_texts   = [queries[qid] for qid in query_ids]
q_vecs    = embed_queries(voyage, q_texts)
print(f"Embedded {len(q_vecs)} queries to dimension {len(q_vecs[0])}.")

## Step 2 — Vector search via `$vectorSearch`

`$vectorSearch` does approximate nearest-neighbour lookup
against the vector index we built in Lab 0. The pipeline
below is the entire interface — embed the query, drop the
vector in as `queryVector`, and ask Atlas for the top-K.

In [ ]:
def dedup_by_doc(rows, top_k):
    """Collapse chunk-level results to one row per parent doc."""
    seen = {}
    for row in rows:
        did = row['doc_id']
        if did not in seen or row['score'] > seen[did]['score']:
            seen[did] = row
    return sorted(seen.values(), key=lambda r: r['score'], reverse=True)[:top_k]


def vector_search(q_vec: list[float], top_k: int = TOP_K) -> list[dict]:
    pipeline = [
        # Stage 1: ANN over our vector index.
        {"$vectorSearch": {
            "index"        : VECTOR_INDEX_NAME,
            "path"         : "embedding",
            "queryVector"  : q_vec,
            "numCandidates": top_k * 20,   # over-fetch for re-ranking
            "limit"        : top_k * 4,
        }},
        # Stage 2: surface the cosine-similarity score.
        {"$addFields": {"score": {"$meta": "vectorSearchScore"}}},
        {"$sort": {"score": -1}},
    ]
    return dedup_by_doc(coll.aggregate(pipeline), top_k)


# Smoke-test on the first query
sample = vector_search(q_vecs[0], top_k=5)
print(f"Vector top-5 for {q_texts[0]!r}:")
for rank, row in enumerate(sample, 1):
    print(f"  {rank}. doc {row['doc_id']:<12} score={row['score']:.3f}  "
          f"{row['title'][:55]!r}")

## Step 3 — Hybrid via `$rankFusion`

Atlas 8.0+ ships `$rankFusion`, a native Reciprocal Rank
Fusion operator that combines multiple ranking pipelines
server-side. You give it the two sub-pipelines (vector and
text) and per-pipeline weights; it computes

$$\text{RRFscore}(d) = \sum_i \frac{w_i}{60 + \text{rank}_i(d)}$$

across both rankings. A single weight `α ∈ [0, 1]` controls
the blend: we pass `weights={vector: α, text: 1-α}`.

In [ ]:
def hybrid_search(q_vec: list[float], q_text: str,
                  alpha: float = 0.8, top_k: int = TOP_K) -> list[dict]:
    """Weighted RRF of vector + lexical via $rankFusion (Atlas 8.0+)."""
    # $rankFusion requires non-zero weights; nudge edges by a tiny epsilon.
    EPS    = 1e-3
    w_vec  = max(alpha,       EPS)
    w_text = max(1.0 - alpha, EPS)
    n      = 100   # per-pipeline candidate depth

    pipeline = [
        {"$rankFusion": {
            "input": {
                "pipelines": {
                    "vector": [
                        {"$vectorSearch": {
                            "index"        : VECTOR_INDEX_NAME,
                            "path"         : "embedding",
                            "queryVector"  : q_vec,
                            "numCandidates": n * 4,
                            "limit"        : n,
                        }},
                    ],
                    "text": [
                        {"$search": {
                            "index": TEXT_INDEX_NAME,
                            "text" : {"path": "text", "query": q_text},
                        }},
                        {"$limit": n},
                    ],
                },
            },
            "combination": {"weights": {"vector": w_vec, "text": w_text}},
        }},
        {"$addFields": {"score": {"$meta": "score"}}},
        {"$limit": top_k * 4},
    ]
    return dedup_by_doc(coll.aggregate(pipeline), top_k)


sample = hybrid_search(q_vecs[0], q_texts[0], alpha=0.8, top_k=5)
print(f"Hybrid (α=0.8) top-5 for {q_texts[0]!r}:")
for rank, row in enumerate(sample, 1):
    print(f"  {rank}. doc {row['doc_id']:<12} score={row['score']:.3f}  "
          f"{row['title'][:55]!r}")

## Step 4 — Lexical, for the comparison

Same as Lab 1, repeated here so the three search functions
sit side-by-side.

In [ ]:
def bm25_search(q_text: str, top_k: int = TOP_K) -> list[dict]:
    pipeline = [
        {"$search": {
            "index": TEXT_INDEX_NAME,
            "text" : {"path": "text", "query": q_text},
        }},
        {"$addFields": {"score": {"$meta": "searchScore"}}},
        {"$limit": top_k * 4},
        {"$sort":  {"score": -1}},
    ]
    return dedup_by_doc(coll.aggregate(pipeline), top_k)

## Step 5 — Evaluate all three on the same queries

Reuse the `query_metrics` we built in Lab 1.

In [ ]:
def query_metrics(ranked, qrels, ks=(5, 10)):
    rel_set = {did for did, s in qrels.items() if s > 0}
    out = {}
    for k in ks:
        top_k = ranked[:k]
        out[f"P@{k}"] = sum(1 for d in top_k if d in rel_set) / k
        out[f"R@{k}"] = len(set(top_k) & rel_set) / max(1, len(rel_set))
        dcg = sum(qrels.get(d, 0) / math.log2(i + 1)
                  for i, d in enumerate(top_k, 1))
        ideal = sorted((s for s in qrels.values() if s > 0), reverse=True)[:k]
        idcg = sum(g / math.log2(i + 1) for i, g in enumerate(ideal, 1))
        out[f"NDCG@{k}"] = dcg / idcg if idcg else 0.0
    out["MRR"] = next(
        (1.0 / i for i, d in enumerate(ranked, 1) if d in rel_set), 0.0,
    )
    cum = hits = 0
    for i, d in enumerate(ranked, 1):
        if d in rel_set:
            hits += 1
            cum  += hits / i
    out["AP"] = cum / len(rel_set) if rel_set else 0.0
    return out


def evaluate(strategy_name, search_fn):
    per_query = []
    for qid, qv, qt in zip(query_ids, q_vecs, q_texts):
        ranked = [r['doc_id'] for r in search_fn(qv, qt)]
        per_query.append(query_metrics(ranked, qrels.get(qid, {})))
    keys = per_query[0].keys()
    agg = {k: sum(q[k] for q in per_query) / len(per_query) for k in keys}
    agg["MAP"] = agg.pop("AP")
    return agg


results = {
    "lexical (BM25)" : evaluate("lexical", lambda v, t: bm25_search(t)),
    "vector"         : evaluate("vector",  lambda v, t: vector_search(v)),
    "hybrid α=0.8"   : evaluate("hybrid",  lambda v, t: hybrid_search(v, t, alpha=0.8)),
}

import pandas as pd
pd.DataFrame(results).T[["P@5", "R@5", "NDCG@5", "NDCG@10", "MRR", "MAP"]].round(3)

## Step 6 — How does `α` move NDCG@10?

The hybrid weight `α` is a knob, not a fixed setting:

- `α = 1.0` — pure vector (same as `vector_search`).
- `α = 0.0` — pure lexical (same as `bm25_search`).
- `α = 0.5` — naïve uniform fusion. *Often the wrong default*.
- `α = 0.8` — vector-favored. Good default on most BEIR data.

Sweep `α` and watch NDCG@10:

In [ ]:
ALPHAS = [0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0]
ndcg_sweep = []
for a in ALPHAS:
    per_query = []
    for qid, qv, qt in zip(query_ids, q_vecs, q_texts):
        ranked = [r['doc_id'] for r in hybrid_search(qv, qt, alpha=a)]
        per_query.append(query_metrics(ranked, qrels.get(qid, {})))
    mean_ndcg = sum(q['NDCG@10'] for q in per_query) / len(per_query)
    ndcg_sweep.append(mean_ndcg)
    print(f"  α={a:.1f}  NDCG@10={mean_ndcg:.3f}")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(ALPHAS, ndcg_sweep, marker='o')
ax.set_xlabel("hybrid α  (0 = lexical, 1 = vector)")
ax.set_ylabel("NDCG@10")
ax.set_title(f"{DATASET} — NDCG@10 vs α ({N_QUERIES} queries)")
ax.grid(alpha=0.3)
plt.show()

## Reading the comparison

Three things to look for:

1. **Vector vs lexical NDCG@10** — embedding-based retrieval
   usually wins on BEIR-style datasets because it handles
   paraphrase and topical overlap. It loses on datasets
   dominated by exact strings (codes, names).

2. **Does hybrid beat both?** Often yes — but only if α is
   weighted toward whichever single mode is stronger. Uniform
   α=0.5 drags vector down when vector is the better signal.

3. **Which metric improves the most?** If vector and hybrid
   lift Recall@10 more than NDCG@10, they're *finding* more
   relevant docs but not ranking them at the top. That's the
   signal that a **cross-encoder reranker** would help — see
   `phase4/rerank.py` for the advanced track.

## What this is *not* telling you

BEIR queries are homogeneous within a dataset — all scientific
claims, or all health questions. A single α tuned on this
dataset will win almost every query. Real production traffic
is messier — a mix of exact-match lookups, single-word topics,
compound questions, and chatty natural language all from the
same users. To know what works on *your* mix, you need *your
own* eval data.

**Next:** open **`03_curate_eval_set.ipynb`**.